In [1]:
# Reproducibility header

import importlib
import os
import random
import subprocess
import sys
import warnings

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'Seed    : {RANDOM_SEED}')

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'fastf1': 'fastf1',
    'requests': 'requests',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed')

import fastf1
import numpy as np
import pandas as pd
import requests

np.random.seed(RANDOM_SEED)

print(f'NumPy   : {np.__version__}')
print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

Python  : 3.14.2
Seed    : 414


All required packages already installed
NumPy   : 2.4.3
fastf1  : 3.8.1
pandas  : 2.3.3


In [2]:
# FastF1 Session Load

cache_path = os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'fastf1_cache')
cache_path = os.path.abspath(cache_path)
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)
session = fastf1.get_session(2024, 'Bahrain', 'R')
session.load()


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]


req            INFO 	No cached data found for session_info. Loading data...


_api           INFO 	Fetching session info data...


req            INFO 	Data has been written to cache!


req            INFO 	No cached data found for driver_info. Loading data...


...

req            INFO 	Data has been written to cache!


req            INFO 	No cached data found for race_control_messages. Loading data...


_api           INFO 	Fetching race control messages...


req            INFO 	Data has been written to cache!


core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


In [3]:
# Load the 2024 Monza Race session and print basic information 

print(f"Session event: {session.event['EventName']} {session.event.year}")
print(f"\nSession: {session.name}")
print(f"\nNumber of laps in the result: {session.results.shape[0]}")
print(f"\nColumn names of the results dataframe: {session.results.columns.tolist()}")
print(f"\nFirst 5 rows of lap data:\n{session.laps.head()}")


Session event: Bahrain Grand Prix 2024

Session: Race

Number of laps in the result: 20

Column names of the results dataframe: ['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']

First 5 rows of lap data:
                    Time Driver DriverNumber                LapTime  \
0 0 days 01:01:37.489000    VER            1 0 days 00:01:37.284000   
1 0 days 01:03:13.785000    VER            1 0 days 00:01:36.296000   
2 0 days 01:04:50.538000    VER            1 0 days 00:01:36.753000   
3 0 days 01:06:27.185000    VER            1 0 days 00:01:36.647000   
4 0 days 01:08:04.358000    VER            1 0 days 00:01:37.173000   

   LapNumber  Stint PitOutTime PitInTime            Sector1Time  \
0        1.0    1.0        NaT       NaT                    NaT   
1        2.0    

In [4]:
# Jolpica API Query https://api.jolpi.ca/ergast/f1/2024/driverstandings/

# Http status code 200 check
response = requests.get("https://api.jolpi.ca/ergast/f1/2024/driverstandings/") 
if response.status_code == 200:
    print("API request successful (code 200).")
else:
    print("API request failed.")

# Number of records returned check
data = response.json()
standings_list = data.get('MRData', {}).get('StandingsTable', {}).get('StandingsLists', [])
if standings_list:
    driver_standings = standings_list[0].get('DriverStandings', [])
    print(f"\nNumber of driver standings records: {len(driver_standings)}")

# First 3 driver entries
if driver_standings:
    print("\nFirst 3 driver standings entries:")
    for entry in driver_standings[:3]:
        driver = entry.get('Driver', {})
        print(f"Driver: {driver.get('givenName', '')} {driver.get('familyName', '')}, Points: {entry.get('points', '')}")



API request successful (code 200).

Number of driver standings records: 24

First 3 driver standings entries:
Driver: Max Verstappen, Points: 437
Driver: Lando Norris, Points: 374
Driver: Charles Leclerc, Points: 356


#### FastF1 session and why you chose it
    > The chosen session was the Bahrian Grand Prix 2024, as it was the recommended session. There is nothing remarkable about the sessions that makes any difference in choosing this or that, so the lack of consequence leads to choosing the first option of all.
### The shapes of both datasets
    > The first dataset is shaped 20x22, while the second one is shaped 24x20.

### One observation about the data structure that was surprising or informative
    > It surprised me that they stored the drivers name and family name as separate rather than a single column with both name and last name, I'm curious as to why this is the case.